# Домашнее задание 4. Аналитические запросы и рекурсия

Практика по семинарам 5–7: агрегация, JOIN, операции над множествами, подзапросы,
обычные и рекурсивные CTE. **20 обязательных задач на 10 баллов**.
Вес задачи зависит от сложности и указан в её заголовке: от 0,25 до 1 балла.
В задаче 9 две обязательные части по 0,25 балла. Дополнительных задач нет.

**Данные.** Используйте PostgreSQL 17 и базу `hse_databases` из учебного окружения версии 2.0.0
или новее: схемы `music_store` (музыкальный магазин) и `film_rental` (прокат фильмов).
Пишите полные имена таблиц со схемой. Во всех задачах достаточно запросов чтения;
создавать или изменять таблицы не требуется. Все задачи независимы.

**Работа с ноутбуком.** В ячейке `%%sql` пишите SQL ниже первой строки;
выполнение — **Shift + Enter**. Один запрос — одна ячейка. В задаче 9 для каждой части
есть отдельная ячейка. Пояснения к рекурсивным запросам заполните в текстовых ячейках.

**Оценивание.** Запрос должен выполняться, возвращать требуемые строки и столбцы
в указанном порядке, соблюдать сортировку и округление. Если способ решения назван явно,
используйте именно его. В остальных случаях допустим любой корректный запрос
без оконных функций. У рекурсивных запросов должно быть обоснованное завершение.

**Числа и даты.** Денежные суммы указаны в денежных единицах исходных баз.
Процент — число от 0 до 100. Округляйте только там и так, как сказано в условии.
Периоды отбора и идентификаторы даны в задачах; самостоятельно подбирать их не нужно.

**Сдача.** Перед отправкой выполните **Run → Restart Kernel and Run All Cells…**,
проверьте отсутствие ошибок и сохраните результаты. Отправьте `.ipynb` на `foxleren@yandex.ru`.
Тема: `ИАД-N ДЗ-4 Фамилия Имя Отчество`; имя файла:
`ИАД-N_ДЗ-4_Фамилия_Имя_Отчество.ipynb`, где N — номер вашей группы.
Настройка окружения описана в README репозитория.

## Памятка

Возле каждой задачи описаны её таблицы: что означает одна строка, какие у неё ключи
и по каким столбцам проходят связи. Полное описание — [db/schema.md](../db/schema.md).

| Приём | Что помнить |
|---|---|
| `COUNT(*)` / `COUNT(column)` | Первый считает строки, второй — только непустые значения столбца. |
| `SUM`, `AVG` | Игнорируют NULL; если значений нет, возвращают NULL. |
| Медиана | `PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY значение)`. Для чётного числа значений берётся среднее двух центральных. |
| `COALESCE(value, 0)` | Заменяет NULL нулём. |
| `ROUND(value, 2)` | Округляет numeric до двух знаков. Для вещественного значения при необходимости используйте `value::numeric`. |
| Дробное деление | `5 / 2` даёт 2, `5 / 2.0` — 2.5. |
| Условная агрегация | `COUNT(*) FILTER (WHERE условие)` или `SUM(CASE WHEN условие THEN 1 ELSE 0 END)`. |
| Год | `EXTRACT(YEAR FROM дата)::integer`. |
| Дата и день | `DATE '2030-04-01' + 1` — следующий день, тоже date. |
| Момент времени | `TIMESTAMPTZ '2030-04-01 00:00:00+03'` — полночь по UTC+03. |
| Множества | `UNION` — объединение без повторов; `INTERSECT` — пересечение; `EXCEPT` — разность. |
| Идентичность | Группируйте объекты по ключам: имена и названия могут совпадать. |

**Рекурсивный CTE** состоит из начальных строк (якоря) и шага, который читает строки
предыдущего шага. Он заканчивается, когда шаг не возвращает новых строк.
`UNION ALL` сохраняет все полученные строки. `ORDER BY` в конечном запросе задаёт порядок вывода:
сам обход нужный порядок не гарантирует.

Пример на другом наборе данных — уменьшение числа до нуля:

```sql
WITH RECURSIVE countdown(n) AS (
    SELECT 3
    UNION ALL
    SELECT
        n - 1
    FROM countdown
    WHERE n > 0
)
SELECT
    n
FROM countdown
ORDER BY n DESC;
```

В иерархии `employee.reports_to` хранит ключ **начальника**, а не подчинённого.
У директора начальника нет. В задачах 18–20 работаем с существующим деревом без циклов.

In [ ]:
import sys
sys.path.insert(0, "../jupyter")
%load_ext sql_magic

## Проверка подключения
Выполните ячейку ниже. Она проверяет доступ к двум нужным схемам.
Если возникла ошибка, сначала проверьте версию окружения по README.

In [ ]:
%%sql
SELECT
    (SELECT COUNT(*) FROM music_store.track) AS music_tracks,
    (SELECT COUNT(*) FROM film_rental.film) AS rental_films;

## Часть 1. Агрегация · music_store

### Задача 1. Портрет музыкального каталога (0,25 балла)

**Устройство нужных таблиц**

`music_store.track`: одна строка — трек в каталоге; ключ — track_id.

| Столбец | Значение |
|---|---|
| `track_id` | идентификатор трека |
| `milliseconds` | длительность в миллисекундах; 60 000 мс = 1 минута |
| `unit_price` | текущая цена трека в каталоге, денежные единицы |

**Что требуется**

Получите общую статистику всех треков каталога одной строкой.
Общую длительность переведите в минуты и округлите до двух знаков после запятой.
Среднюю цену также округлите до двух знаков. Минимальную и максимальную цены не округляйте.

**Столбцы результата, по порядку:** track_count — число треков; total_minutes — общая длительность в минутах; min_price — минимальная цена; avg_price — средняя цена; max_price — максимальная цена

**Сортировка:** Не требуется: результат состоит из одной строки.

In [ ]:
%%sql
-- Задача 1. Напишите запрос ниже.

### Задача 2. Покупки по странам (0,5 балла)

**Устройство нужных таблиц**

`music_store.invoice`: одна строка — счёт покупателя — одна покупка; ключ — invoice_id.

| Столбец | Значение |
|---|---|
| `invoice_id` | идентификатор счёта |
| `billing_country` | страна в счёте на момент покупки |
| `total` | полная сумма счёта, денежные единицы; не сумма одной строки счёта |

**Что требуется**

Для каждой страны из счетов посчитайте число покупок, общую выручку,
среднюю и медианную сумму покупки. Используйте страну из счёта,
а не текущую страну покупателя. Среднюю и медианную суммы округлите до двух знаков.
Для медианы используйте PERCENTILE_CONT(0.5): это центральное значение отсортированных сумм,
а при чётном числе счетов — среднее двух центральных значений.

**Способ.** Для медианы используйте PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY …).

**Столбцы результата, по порядку:** billing_country — страна; invoice_count — число счетов; revenue — сумма счетов; avg_invoice — средняя сумма счёта; median_invoice — медианная сумма счёта

**Сортировка:** revenue по убыванию, затем billing_country по возрастанию.

In [ ]:
%%sql
-- Задача 2. Напишите запрос ниже.

### Задача 3. Крупные рынки за выбранный период (0,5 балла)

**Устройство нужных таблиц**

`music_store.invoice`: одна строка — счёт покупателя — одна покупка; ключ — invoice_id.

| Столбец | Значение |
|---|---|
| `invoice_id` | идентификатор счёта |
| `invoice_date` | дата покупки, тип date |
| `billing_country` | страна в счёте на момент покупки |
| `total` | полная сумма счёта, денежные единицы; не сумма одной строки счёта |

**Что требуется**

Рассмотрите покупки с 1 января 2023 года включительно до 1 января 2025 года не включительно.
Найдите страны, в которых за этот период было не меньше 10 покупок и средняя сумма покупки
строго больше 5 денежных единиц. Порог проверяйте по неокруглённому среднему,
а для вывода округлите его до двух знаков.

**Столбцы результата, по порядку:** billing_country — страна; invoice_count — число покупок за период; avg_invoice — средняя сумма покупки за период

**Сортировка:** avg_invoice по убыванию, затем billing_country по возрастанию.

In [ ]:
%%sql
-- Задача 3. Напишите запрос ниже.

### Задача 4. США и остальные страны по годам (0,5 балла)

**Устройство нужных таблиц**

`music_store.invoice`: одна строка — счёт покупателя — одна покупка; ключ — invoice_id.

| Столбец | Значение |
|---|---|
| `invoice_id` | идентификатор счёта |
| `invoice_date` | дата покупки, тип date |
| `billing_country` | страна в счёте на момент покупки |
| `total` | полная сумма счёта, денежные единицы; не сумма одной строки счёта |

**Что требуется**

Для каждого года, встречающегося в счетах, выведите число покупок и выручку отдельно
для USA и для всех остальных стран. Если в одной группе за год нет покупок, выводите ноль.
В этих данных страна счёта заполнена. Используйте условную агрегацию через FILTER или CASE.

**Столбцы результата, по порядку:** sales_year — год; usa_count — число покупок из USA; usa_revenue — их сумма; other_count — число остальных покупок; other_revenue — их сумма

**Сортировка:** sales_year по возрастанию.

In [ ]:
%%sql
-- Задача 4. Напишите запрос ниже.

### Задача 5. Заполненность карточек покупателей (0,25 балла)

**Устройство нужных таблиц**

`music_store.customer`: одна строка — покупатель; ключ — customer_id.

| Столбец | Значение |
|---|---|
| `customer_id` | идентификатор покупателя |
| `country` | страна в текущей карточке покупателя |
| `company` | название компании; NULL, если не указана; пустые строки уже заменены на NULL |

**Что требуется**

Для каждой страны покупателей посчитайте общее число покупателей, число карточек
с указанной компанией и без неё, а также процент карточек с компанией.
Процент выведите по шкале от 0 до 100, округлённым до двух знаков.

**Столбцы результата, по порядку:** country — страна; customer_count — всего покупателей; with_company — компания указана; without_company — компания не указана; company_pct — процент с компанией

**Сортировка:** country по возрастанию, NULL в конце.

In [ ]:
%%sql
-- Задача 5. Напишите запрос ниже.

## Часть 2. JOIN и операции над множествами · film_rental

### Задача 6. Выдачи фильмов за день (0,5 балла)

**Устройство нужных таблиц**

`film_rental.rental`: одна строка — отдельная выдача копии клиенту; ключ — rental_id.

| Столбец | Значение |
|---|---|
| `rental_id` | идентификатор выдачи |
| `inventory_id` | ссылка на inventory.inventory_id; копию могут выдавать многократно |
| `customer_id` | ссылка на customer.customer_id |
| `rental_date` | момент выдачи, timestamptz |
| `return_date` | момент возврата, timestamptz; NULL — не возвращена |

`film_rental.inventory`: одна строка — физическая копия фильма в пункте проката; ключ — inventory_id.

| Столбец | Значение |
|---|---|
| `inventory_id` | идентификатор копии |
| `film_id` | ссылка на film.film_id; у фильма может быть несколько копий |

`film_rental.film`: одна строка — фильм; ключ — film_id; фильм и физическая копия — разные объекты.

| Столбец | Значение |
|---|---|
| `film_id` | идентификатор фильма |
| `title` | название |

`film_rental.customer`: одна строка — клиент проката; ключ — customer_id.

| Столбец | Значение |
|---|---|
| `customer_id` | идентификатор клиента |
| `first_name` | имя |
| `last_name` | фамилия |

**Что требуется**

Выведите все выдачи с 1 августа 2022 года 00:00 включительно
до 2 августа 2022 года 00:00 не включительно по московскому времени (UTC+03).
Для каждой выдачи покажите клиента и название фильма. Невозвращённые выдачи сохраняйте;
дату возврата оставляйте NULL. Одна строка результата — одна выдача.

**Столбцы результата, по порядку:** rental_id — выдача; customer_id — клиент; first_name — имя; last_name — фамилия; title — фильм; rental_date — момент выдачи; return_date — момент возврата

**Сортировка:** rental_date по возрастанию, затем rental_id по возрастанию.

In [ ]:
%%sql
-- Задача 6. Напишите запрос ниже.

### Задача 7. Наличие фильмов в двух пунктах проката (0,5 балла)

**Устройство нужных таблиц**

`film_rental.film`: одна строка — фильм; ключ — film_id; фильм и физическая копия — разные объекты.

| Столбец | Значение |
|---|---|
| `film_id` | идентификатор фильма |
| `title` | название |

`film_rental.inventory`: одна строка — физическая копия фильма в пункте проката; ключ — inventory_id.

| Столбец | Значение |
|---|---|
| `inventory_id` | идентификатор копии |
| `film_id` | ссылка на film.film_id; у фильма может быть несколько копий |
| `store_id` | идентификатор пункта проката |

**Что требуется**

Для каждого фильма посчитайте число физических копий во всей сети и отдельно
в пунктах проката с store_id = 1 и store_id = 2. Сохраните все фильмы, включая фильмы без копий.
Отсутствие копий обозначайте нулём. Речь о наличии в каталоге inventory, а не о том,
свободна ли копия сейчас.

**Столбцы результата, по порядку:** film_id — фильм; title — название; copy_count — копий всего; store_1_count — копий в пункте 1; store_2_count — копий в пункте 2

**Сортировка:** film_id по возрастанию.

In [ ]:
%%sql
-- Задача 7. Напишите запрос ниже.

### Задача 8. Актёры и копии в одной карточке фильма (0,75 балла)

**Устройство нужных таблиц**

`film_rental.film`: одна строка — фильм; ключ — film_id; фильм и физическая копия — разные объекты.

| Столбец | Значение |
|---|---|
| `film_id` | идентификатор фильма |
| `title` | название |

`film_rental.film_actor`: одна строка — участие актёра в фильме; ключ — пара (film_id, actor_id).

| Столбец | Значение |
|---|---|
| `film_id` | ссылка на film.film_id |
| `actor_id` | идентификатор актёра; у фильма может быть несколько актёров |

`film_rental.inventory`: одна строка — физическая копия фильма в пункте проката; ключ — inventory_id.

| Столбец | Значение |
|---|---|
| `inventory_id` | идентификатор копии |
| `film_id` | ссылка на film.film_id; у фильма может быть несколько копий |

**Что требуется**

Для каждого фильма выведите число разных актёров и число физических копий.
Сохраните все фильмы: при отсутствии актёров или копий соответствующее число равно нулю.
Учитывайте, что фильм одновременно может иметь несколько актёров и несколько копий.

**Столбцы результата, по порядку:** film_id — фильм; title — название; actor_count — число актёров; copy_count — число копий

**Сортировка:** film_id по возрастанию.

In [ ]:
%%sql
-- Задача 8. Напишите запрос ниже.

### Задача 9. Любители комедий и фильмов ужасов (0,5 балла)

**Устройство нужных таблиц**

`film_rental.rental`: одна строка — отдельная выдача копии клиенту; ключ — rental_id.

| Столбец | Значение |
|---|---|
| `rental_id` | идентификатор выдачи |
| `inventory_id` | ссылка на inventory.inventory_id; копию могут выдавать многократно |
| `customer_id` | ссылка на customer.customer_id |
| `rental_date` | момент выдачи, timestamptz |

`film_rental.inventory`: одна строка — физическая копия фильма в пункте проката; ключ — inventory_id.

| Столбец | Значение |
|---|---|
| `inventory_id` | идентификатор копии |
| `film_id` | ссылка на film.film_id; у фильма может быть несколько копий |

`film_rental.film_category`: одна строка — связь фильма с жанром; ключ — пара (film_id, category_id).

| Столбец | Значение |
|---|---|
| `film_id` | ссылка на film.film_id |
| `category_id` | ссылка на category.category_id |

`film_rental.category`: одна строка — жанр фильма; ключ — category_id.

| Столбец | Значение |
|---|---|
| `category_id` | идентификатор жанра |
| `name` | название жанра; регистр важен при сравнении |

**Что требуется**

Рассмотрите выдачи за июль 2022 года: с 1 июля 00:00 включительно до 1 августа 00:00
не включительно по московскому времени (UTC+03). Множество A — клиенты, бравшие фильмы
жанра Comedy за этот период; множество B — клиенты, бравшие фильмы жанра Horror за тот же период.

**9.1 (0,25 балла).** Получите клиентов, входящих хотя бы в одно из множеств, через UNION.

**9.2 (0,25 балла).** Получите клиентов, входящих в оба множества, через INTERSECT.

В каждом ответе клиент должен встречаться один раз независимо от числа выдач.

**Способ.** В 9.1 обязателен UNION, в 9.2 — INTERSECT.

**Столбцы результата, по порядку:** В каждой части один столбец customer_id — идентификатор клиента.

**Сортировка:** В каждой части customer_id по возрастанию.

In [ ]:
%%sql
-- Задача 9.1. Напишите запрос ниже.

In [ ]:
%%sql
-- Задача 9.2. Напишите запрос ниже.

### Задача 10. Комедии без фильмов ужасов (0,25 балла)

**Устройство нужных таблиц**

`film_rental.rental`: одна строка — отдельная выдача копии клиенту; ключ — rental_id.

| Столбец | Значение |
|---|---|
| `rental_id` | идентификатор выдачи |
| `inventory_id` | ссылка на inventory.inventory_id; копию могут выдавать многократно |
| `customer_id` | ссылка на customer.customer_id |
| `rental_date` | момент выдачи, timestamptz |

`film_rental.inventory`: одна строка — физическая копия фильма в пункте проката; ключ — inventory_id.

| Столбец | Значение |
|---|---|
| `inventory_id` | идентификатор копии |
| `film_id` | ссылка на film.film_id; у фильма может быть несколько копий |

`film_rental.film_category`: одна строка — связь фильма с жанром; ключ — пара (film_id, category_id).

| Столбец | Значение |
|---|---|
| `film_id` | ссылка на film.film_id |
| `category_id` | ссылка на category.category_id |

`film_rental.category`: одна строка — жанр фильма; ключ — category_id.

| Столбец | Значение |
|---|---|
| `category_id` | идентификатор жанра |
| `name` | название жанра; регистр важен при сравнении |

**Что требуется**

Найдите клиентов, которые в июле 2022 года брали хотя бы один фильм жанра Comedy,
но не брали ни одного фильма жанра Horror. Период: с 1 июля 00:00 включительно
до 1 августа 00:00 не включительно по московскому времени (UTC+03).
Выдачи вне этого периода не учитывайте. Клиента выведите один раз.

**Способ.** Используйте EXCEPT.

**Столбцы результата, по порядку:** customer_id — идентификатор клиента

**Сортировка:** customer_id по возрастанию.

In [ ]:
%%sql
-- Задача 10. Напишите запрос ниже.

## Часть 3. Подзапросы и обычные CTE · music_store

### Задача 11. Покупки дороже среднего (0,5 балла)

**Устройство нужных таблиц**

`music_store.invoice`: одна строка — счёт покупателя — одна покупка; ключ — invoice_id.

| Столбец | Значение |
|---|---|
| `invoice_id` | идентификатор счёта |
| `total` | полная сумма счёта, денежные единицы; не сумма одной строки счёта |

**Что требуется**

Выведите счета, сумма которых строго выше средней суммы всех счетов за всю историю.
Рядом покажите общее среднее и превышение суммы счёта над ним.
Для отбора используйте неокруглённое среднее. Среднее и разницу округлите
независимо до двух знаков: разницу вычисляйте до округления её компонентов.

**Способ.** Используйте скалярный подзапрос для вычисления среднего.

**Столбцы результата, по порядку:** invoice_id — счёт; total — его сумма; avg_total — средняя сумма всех счетов; diff — превышение над средним

**Сортировка:** total по убыванию, затем invoice_id по возрастанию.

In [ ]:
%%sql
-- Задача 11. Напишите запрос ниже.

### Задача 12. Треки длиннее среднего по своему альбому (0,5 балла)

**Устройство нужных таблиц**

`music_store.track`: одна строка — трек в каталоге; ключ — track_id.

| Столбец | Значение |
|---|---|
| `track_id` | идентификатор трека |
| `name` | название; не уникально |
| `album_id` | альбом, ссылка на album.album_id; может быть NULL |
| `milliseconds` | длительность в миллисекундах; 60 000 мс = 1 минута |

**Что требуется**

Выведите треки, длительность которых строго больше средней длительности треков
того же альбома. Треки без album_id исключите. Среднюю длительность и превышение
округлите до двух знаков, единицы — миллисекунды. Сравнивайте неокруглённые значения.

**Способ.** Используйте коррелированный подзапрос.

**Столбцы результата, по порядку:** track_id — трек; name — название; album_id — альбом; milliseconds — длительность; avg_album_ms — средняя длительность в альбоме; diff_ms — превышение

**Сортировка:** album_id по возрастанию, затем milliseconds по убыванию, затем track_id по возрастанию.

In [ ]:
%%sql
-- Задача 12. Напишите запрос ниже.

### Задача 13. Непроданные треки (0,25 балла)

**Устройство нужных таблиц**

`music_store.track`: одна строка — трек в каталоге; ключ — track_id.

| Столбец | Значение |
|---|---|
| `track_id` | идентификатор трека |
| `name` | название; не уникально |

`music_store.invoice_line`: одна строка — строка счёта — покупка определённого трека; ключ — invoice_line_id.

| Столбец | Значение |
|---|---|
| `invoice_line_id` | идентификатор строки |
| `track_id` | ссылка на track.track_id; один трек может покупаться много раз |

**Что требуется**

Найдите все треки каталога, для которых нет ни одной строки покупки за всю историю.
Выведите каждый такой трек один раз.

**Способ.** Используйте NOT EXISTS.

**Столбцы результата, по порядку:** track_id — трек; name — название

**Сортировка:** track_id по возрастанию.

In [ ]:
%%sql
-- Задача 13. Напишите запрос ниже.

### Задача 14. Плейлисты и продажи их треков (0,75 балла)

**Устройство нужных таблиц**

`music_store.playlist`: одна строка — плейлист; ключ — playlist_id.

| Столбец | Значение |
|---|---|
| `playlist_id` | идентификатор плейлиста |
| `name` | название; у разных плейлистов может совпадать |

`music_store.playlist_track`: одна строка — включение трека в плейлист; ключ — пара (playlist_id, track_id).

| Столбец | Значение |
|---|---|
| `playlist_id` | ссылка на playlist.playlist_id |
| `track_id` | ссылка на track.track_id |

`music_store.track`: одна строка — трек в каталоге; ключ — track_id.

| Столбец | Значение |
|---|---|
| `track_id` | идентификатор трека |
| `milliseconds` | длительность в миллисекундах; 60 000 мс = 1 минута |

`music_store.invoice_line`: одна строка — строка счёта — покупка определённого трека; ключ — invoice_line_id.

| Столбец | Значение |
|---|---|
| `invoice_line_id` | идентификатор строки |
| `track_id` | ссылка на track.track_id; один трек может покупаться много раз |

**Что требуется**

Для каждого плейлиста посчитайте число входящих в него треков, их общую длительность
в минутах и число разных треков плейлиста, купленных хотя бы один раз за всю историю.
Проданный трек учитывается один раз независимо от числа его покупок.
Сохраните пустые плейлисты: все три показателя для них равны нулю.
Длительность округлите до двух знаков после суммирования.
Плейлисты с одинаковыми названиями считайте разными объектами.

**Способ.** Используйте хотя бы один обычный CTE (WITH).

**Столбцы результата, по порядку:** playlist_id — плейлист; name — название; track_count — число треков; total_minutes — общая длительность; sold_track_count — число разных проданных треков

**Сортировка:** playlist_id по возрастанию.

In [ ]:
%%sql
-- Задача 14. Напишите запрос ниже.

### Задача 15. Покупатели выше среднего по своей стране (0,75 балла)

**Устройство нужных таблиц**

`music_store.customer`: одна строка — покупатель; ключ — customer_id.

| Столбец | Значение |
|---|---|
| `customer_id` | идентификатор покупателя |
| `first_name` | имя |
| `last_name` | фамилия |
| `country` | страна в текущей карточке покупателя |

`music_store.invoice`: одна строка — счёт покупателя — одна покупка; ключ — invoice_id.

| Столбец | Значение |
|---|---|
| `invoice_id` | идентификатор счёта |
| `customer_id` | покупатель, ссылка на customer.customer_id |
| `total` | полная сумма счёта, денежные единицы; не сумма одной строки счёта |

**Что требуется**

Сначала определите расходы каждого покупателя за всю историю; при отсутствии счетов
расходы равны нулю. Затем для каждой страны из карточек покупателей найдите среднее
этих индивидуальных расходов, включая покупателей с нулевыми расходами.
Выведите покупателей, чьи расходы строго выше среднего по своей стране.
Среднее для вывода округлите до двух знаков, для сравнения не округляйте.
В этих данных страна покупателя заполнена.

**Способ.** Используйте не меньше двух обычных CTE.

**Столбцы результата, по порядку:** customer_id — покупатель; first_name — имя; last_name — фамилия; country — страна; total_spent — расходы; avg_country_spent — средние расходы покупателя в стране

**Сортировка:** country по возрастанию, затем total_spent по убыванию, затем customer_id по возрастанию.

In [ ]:
%%sql
-- Задача 15. Напишите запрос ниже.

## Часть 4. Рекурсивные CTE · music_store

### Задача 16. Последовательность месяцев (0,25 балла)

**Нужные таблицы.** Таблицы не нужны: запрос строит последовательность самостоятельно.

**Что требуется**

Постройте последовательность целых чисел от 1 до 12 включительно.
Начальная часть рекурсивного CTE должна возвращать только 1, а каждый следующий шаг —
увеличивать число на 1. Каждое число выведите один раз. Готовый список из 12 чисел
и generate_series не используйте.

**Способ.** Используйте WITH RECURSIVE. Готовые генераторы последовательностей запрещены.

**Столбцы результата, по порядку:** month_no — номер месяца

**Сортировка:** month_no по возрастанию.

In [ ]:
%%sql
-- Задача 16. Напишите запрос ниже.

**Пояснение рекурсии**

Якорь: …

Один шаг рекурсии: …

Почему выполнение заканчивается: …

### Задача 17. Календарь ежедневных продаж (0,5 балла)

**Устройство нужных таблиц**

`music_store.invoice`: одна строка — счёт покупателя — одна покупка; ключ — invoice_id.

| Столбец | Значение |
|---|---|
| `invoice_id` | идентификатор счёта |
| `invoice_date` | дата покупки, тип date |
| `total` | полная сумма счёта, денежные единицы; не сумма одной строки счёта |

**Что требуется**

Постройте календарь с 1 по 31 января 2021 года включительно рекурсивным CTE.
Для каждого дня покажите число счетов и их общую сумму.
Дни без покупок тоже должны присутствовать: оба показателя равны нулю.
Начните с DATE '2021-01-01' и на каждом шаге прибавляйте один день.
generate_series и готовый список дат не используйте.

**Способ.** Используйте WITH RECURSIVE для календаря.

**Столбцы результата, по порядку:** sales_date — дата, тип date; invoice_count — число счетов; revenue — сумма счетов

**Сортировка:** sales_date по возрастанию.

In [ ]:
%%sql
-- Задача 17. Напишите запрос ниже.

**Пояснение рекурсии**

Якорь: …

Один шаг рекурсии: …

Почему выполнение заканчивается: …

### Задача 18. Все подчинённые руководителя (0,5 балла)

**Устройство нужных таблиц**

`music_store.employee`: одна строка — сотрудник; ключ — employee_id.

| Столбец | Значение |
|---|---|
| `employee_id` | идентификатор сотрудника |
| `first_name` | имя |
| `last_name` | фамилия |
| `reports_to` | идентификатор непосредственного начальника в этой же таблице; NULL у директора |

**Что требуется**

Найдите всех прямых и косвенных подчинённых сотрудника с employee_id = 1.
Самого сотрудника 1 не выводите. Для непосредственных подчинённых глубина равна 1,
для их подчинённых — 2 и так далее. Выведите каждого подчинённого один раз.
Иерархия в учебных данных не содержит циклов.

**Способ.** Используйте WITH RECURSIVE; фиксированную цепочку JOIN вместо рекурсии не используйте.

**Столбцы результата, по порядку:** employee_id — подчинённый; first_name — имя; last_name — фамилия; depth — глубина относительно сотрудника 1

**Сортировка:** depth по возрастанию, затем employee_id по возрастанию.

In [ ]:
%%sql
-- Задача 18. Напишите запрос ниже.

**Пояснение рекурсии**

Якорь: …

Один шаг рекурсии: …

Почему выполнение заканчивается: …

### Задача 19. Цепочки начальников всех сотрудников (0,5 балла)

**Устройство нужных таблиц**

`music_store.employee`: одна строка — сотрудник; ключ — employee_id.

| Столбец | Значение |
|---|---|
| `employee_id` | идентификатор сотрудника |
| `first_name` | имя |
| `last_name` | фамилия |
| `reports_to` | идентификатор непосредственного начальника в этой же таблице; NULL у директора |

**Что требуется**

Для каждого сотрудника найдите всех его прямых и косвенных начальников до директора включительно.
Одна строка результата — пара «исходный сотрудник — его начальник».
Непосредственный начальник имеет distance = 1, его начальник — distance = 2 и так далее.
Сотрудник не является начальником самому себе. Сотрудники без начальников не дают строк
в роли исходного сотрудника, но могут встречаться в роли начальника.
Каждую пару выведите один раз. Иерархия не содержит циклов.

**Способ.** Используйте WITH RECURSIVE.

**Столбцы результата, по порядку:** employee_id — исходный сотрудник; first_name — его имя; last_name — его фамилия; manager_id — начальник; manager_first_name — имя начальника; manager_last_name — фамилия начальника; distance — число переходов от исходного сотрудника к начальнику

**Сортировка:** employee_id по возрастанию, затем distance по возрастанию, затем manager_id по возрастанию.

In [ ]:
%%sql
-- Задача 19. Напишите запрос ниже.

**Пояснение рекурсии**

Якорь: …

Один шаг рекурсии: …

Почему выполнение заканчивается: …

### Задача 20. Выручка команды каждого сотрудника (1 балл)

**Устройство нужных таблиц**

`music_store.employee`: одна строка — сотрудник; ключ — employee_id.

| Столбец | Значение |
|---|---|
| `employee_id` | идентификатор сотрудника |
| `first_name` | имя |
| `last_name` | фамилия |
| `reports_to` | идентификатор непосредственного начальника в этой же таблице; NULL у директора |

`music_store.customer`: одна строка — покупатель; ключ — customer_id.

| Столбец | Значение |
|---|---|
| `customer_id` | идентификатор покупателя |
| `support_rep_id` | обслуживающий сотрудник, ссылка на employee.employee_id; может быть NULL |

`music_store.invoice`: одна строка — счёт покупателя — одна покупка; ключ — invoice_id.

| Столбец | Значение |
|---|---|
| `invoice_id` | идентификатор счёта |
| `customer_id` | покупатель, ссылка на customer.customer_id |
| `total` | полная сумма счёта, денежные единицы; не сумма одной строки счёта |

**Что требуется**

Для каждого сотрудника найдите выручку его команды за всю историю.
Команда состоит из самого сотрудника и всех его прямых и косвенных подчинённых.
Покупатель относится к сотруднику через customer.support_rep_id.
Выручка команды — сумма invoice.total по счетам покупателей её участников.

Счёт учитывается один раз в команде каждого соответствующего руководителя.
Один и тот же счёт может входить в показатели разных сотрудников: например, агента и его начальника.
Выведите всех сотрудников, в том числе без покупателей и продаж, с нулевой выручкой.
Иерархия не содержит циклов; у каждого сотрудника не больше одного непосредственного начальника.

**Способ.** Используйте WITH RECURSIVE для формирования команд.

**Столбцы результата, по порядку:** employee_id — сотрудник; first_name — имя; last_name — фамилия; team_revenue — выручка его команды

**Сортировка:** team_revenue по убыванию, затем employee_id по возрастанию.

In [ ]:
%%sql
-- Задача 20. Напишите запрос ниже.

**Пояснение рекурсии**

Якорь: …

Один шаг рекурсии: …

Почему выполнение заканчивается: …